# V2 — Calibration و threshold نهایی A2-MP

این notebook از probability سطح ویدئو که در notebook 21 ذخیره شده استفاده می‌کند. مدل یا ویدیو را دوباره آموزش/پردازش نمی‌کند. calibration فقط پس از تثبیت مدل انجام می‌شود و زمان رخداد هرگز ورودی آن نیست.

In [11]:
from __future__ import annotations

from pathlib import Path
import json

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, average_precision_score, brier_score_loss, confusion_matrix, f1_score, precision_score, recall_score, roc_auc_score
from sklearn.model_selection import StratifiedKFold
import torch
from torch.nn import functional as F

DATA_ROOT = Path(r'P:\\NexarCollisionData')
INFERENCE_DIR = DATA_ROOT / 'inference_v2'
VIDEO_PREDICTIONS_PATH = INFERENCE_DIR / 'a2_multipos_validation_sliding_video_predictions.csv'
SLIDING_METRICS_PATH = INFERENCE_DIR / 'a2_multipos_validation_sliding_metrics.json'
CALIBRATED_PREDICTIONS_PATH = INFERENCE_DIR / 'a2_multipos_calibrated_validation_predictions.csv'
CALIBRATION_REPORT_PATH = INFERENCE_DIR / 'a2_multipos_calibration_report.json'
RELIABILITY_PLOT_PATH = INFERENCE_DIR / 'a2_multipos_reliability_diagram.png'
THRESHOLD_CURVE_PATH = INFERENCE_DIR / 'a2_multipos_calibrated_threshold_curve.csv'

MINIMUM_ACCIDENT_RECALL = 0.80
CALIBRATION_FOLDS = 5
CALIBRATION_SEED = 42
NUM_BINS = 10

assert VIDEO_PREDICTIONS_PATH.exists(), 'Run notebook 21 first.'
assert SLIDING_METRICS_PATH.exists(), 'Run notebook 21 first.'

In [12]:
sliding_metrics = json.loads(SLIDING_METRICS_PATH.read_text(encoding='utf-8'))
selected_aggregation = sliding_metrics['selected_aggregation']
raw_selected_threshold = float(sliding_metrics['selected_threshold_by_validation_f1_under_recall_constraint'])

video_predictions = pd.read_csv(VIDEO_PREDICTIONS_PATH).copy()
video_predictions['label'] = video_predictions['label'].astype(int)
assert len(video_predictions) == 120
assert video_predictions.groupby('label').size().to_dict() == {0: 60, 1: 60}
assert selected_aggregation in video_predictions.columns

labels = video_predictions['label'].to_numpy(dtype=int)
raw_probabilities = video_predictions[selected_aggregation].to_numpy(dtype=float)
assert np.isfinite(raw_probabilities).all()
assert ((raw_probabilities > 0.0) & (raw_probabilities < 1.0)).all()

print({'aggregation': selected_aggregation, 'raw_selected_threshold': raw_selected_threshold})
display(video_predictions[['video_id', 'label', selected_aggregation, 'max_window_start', 'max_window_end']].head())

{'aggregation': 'top3_mean', 'raw_selected_threshold': 0.4}


,video_id,label,top3_mean,max_window_start,max_window_end
0,14,1,0.504325,32.5,37.5
1,29,1,0.551898,17.5,22.5
2,31,1,0.888302,15.0,20.0
3,32,1,0.840568,27.5,32.5
4,56,1,0.469058,5.0,10.0


In [13]:
def probabilities_to_logits(probabilities: np.ndarray) -> np.ndarray:
    probabilities = np.clip(np.asarray(probabilities, dtype=np.float64), 1e-6, 1.0 - 1e-6)
    return np.log(probabilities / (1.0 - probabilities))

def logits_to_probabilities(logits: np.ndarray) -> np.ndarray:
    return 1.0 / (1.0 + np.exp(-np.asarray(logits, dtype=np.float64)))

def fit_temperature(probabilities: np.ndarray, y_true: np.ndarray) -> float:
    logits = torch.as_tensor(probabilities_to_logits(probabilities), dtype=torch.float32)
    targets = torch.as_tensor(y_true, dtype=torch.float32)
    log_temperature = torch.zeros(1, dtype=torch.float32, requires_grad=True)
    optimizer = torch.optim.LBFGS([log_temperature], lr=0.1, max_iter=100, line_search_fn='strong_wolfe')

    def closure():
        optimizer.zero_grad()
        temperature = log_temperature.exp().clamp(0.05, 20.0)
        loss = F.binary_cross_entropy_with_logits(logits / temperature, targets)
        loss.backward()
        return loss

    optimizer.step(closure)
    return float(log_temperature.detach().exp().clamp(0.05, 20.0).item())

def apply_temperature(probabilities: np.ndarray, temperature: float) -> np.ndarray:
    return logits_to_probabilities(probabilities_to_logits(probabilities) / temperature)

def expected_calibration_error(y_true: np.ndarray, probabilities: np.ndarray, bins: int = NUM_BINS) -> tuple[float, pd.DataFrame]:
    edges = np.linspace(0.0, 1.0, bins + 1)
    records = []
    ece = 0.0
    for index in range(bins):
        lower, upper = edges[index], edges[index + 1]
        in_bin = (probabilities >= lower) & ((probabilities < upper) if index < bins - 1 else (probabilities <= upper))
        count = int(in_bin.sum())
        if count:
            mean_probability = float(probabilities[in_bin].mean())
            empirical_frequency = float(y_true[in_bin].mean())
            gap = abs(mean_probability - empirical_frequency)
            ece += (count / len(y_true)) * gap
        else:
            mean_probability, empirical_frequency, gap = np.nan, np.nan, np.nan
        records.append({
            'bin_lower': lower, 'bin_upper': upper, 'count': count,
            'mean_probability': mean_probability, 'empirical_frequency': empirical_frequency, 'absolute_gap': gap,
        })
    return float(ece), pd.DataFrame(records)

def binary_metrics(y_true: np.ndarray, probabilities: np.ndarray, threshold: float) -> dict:
    predictions = (probabilities >= threshold).astype(int)
    return {
        'threshold': float(threshold),
        'accuracy': float(accuracy_score(y_true, predictions)),
        'precision': float(precision_score(y_true, predictions, zero_division=0)),
        'recall': float(recall_score(y_true, predictions, zero_division=0)),
        'f1': float(f1_score(y_true, predictions, zero_division=0)),
        'roc_auc': float(roc_auc_score(y_true, probabilities)),
        'pr_auc': float(average_precision_score(y_true, probabilities)),
        'brier_score': float(brier_score_loss(y_true, probabilities)),
        'confusion_matrix': confusion_matrix(y_true, predictions).tolist(),
    }

def select_threshold(y_true: np.ndarray, probabilities: np.ndarray) -> tuple[float, pd.DataFrame]:
    curve = pd.DataFrame([
        binary_metrics(y_true, probabilities, float(threshold))
        for threshold in np.round(np.arange(0.05, 0.951, 0.01), 2)
    ])
    safe_curve = curve.loc[curve['recall'].ge(MINIMUM_ACCIDENT_RECALL)]
    selection_pool = safe_curve if len(safe_curve) else curve
    selected = selection_pool.sort_values(['f1', 'recall', 'precision'], ascending=False).iloc[0]
    return float(selected['threshold']), curve

In [14]:
cross_fitted_probabilities = np.full_like(raw_probabilities, np.nan, dtype=float)
cross_fitted_temperatures = []
folds = StratifiedKFold(n_splits=CALIBRATION_FOLDS, shuffle=True, random_state=CALIBRATION_SEED)
for fold_index, (calibration_indices, heldout_indices) in enumerate(folds.split(raw_probabilities, labels), start=1):
    temperature = fit_temperature(raw_probabilities[calibration_indices], labels[calibration_indices])
    cross_fitted_probabilities[heldout_indices] = apply_temperature(raw_probabilities[heldout_indices], temperature)
    cross_fitted_temperatures.append({'fold': fold_index, 'temperature': temperature, 'calibration_samples': int(len(calibration_indices),), 'heldout_samples': int(len(heldout_indices))})

assert np.isfinite(cross_fitted_probabilities).all()
final_temperature = fit_temperature(raw_probabilities, labels)
final_calibrated_probabilities = apply_temperature(raw_probabilities, final_temperature)
grid_selected_threshold, calibrated_threshold_curve = select_threshold(labels, final_calibrated_probabilities)
# Temperature scaling is strictly monotonic. Map the already selected raw operating point exactly instead of changing decisions because of a 0.01 threshold grid.
calibrated_selected_threshold = float(apply_temperature(np.asarray([raw_selected_threshold]), final_temperature)[0])
raw_ece, raw_reliability = expected_calibration_error(labels, raw_probabilities)
cross_fitted_ece, cross_fitted_reliability = expected_calibration_error(labels, cross_fitted_probabilities)
final_fitted_ece, final_fitted_reliability = expected_calibration_error(labels, final_calibrated_probabilities)

raw_metrics_at_raw_threshold = binary_metrics(labels, raw_probabilities, raw_selected_threshold)
calibrated_metrics_at_selected_threshold = binary_metrics(labels, final_calibrated_probabilities, calibrated_selected_threshold)
assert np.array_equal(raw_probabilities >= raw_selected_threshold, final_calibrated_probabilities >= calibrated_selected_threshold)
print({'final_temperature': final_temperature, 'calibrated_threshold_equivalent_to_raw_threshold': calibrated_selected_threshold, 'grid_selected_threshold_for_reference': grid_selected_threshold})
display(pd.DataFrame(cross_fitted_temperatures))

{'final_temperature': 1.4084302186965942, 'calibrated_threshold_equivalent_to_raw_threshold': 0.4285218762661993, 'grid_selected_threshold_for_reference': 0.43}


,fold,temperature,calibration_samples,heldout_samples
0,1,1.328619,96,24
1,2,1.240344,96,24
2,3,1.278515,96,24
3,4,1.774773,96,24
4,5,1.504098,96,24


In [15]:
video_predictions['raw_video_probability'] = raw_probabilities
video_predictions['cross_fitted_calibrated_probability'] = cross_fitted_probabilities
video_predictions['final_calibrated_probability'] = final_calibrated_probabilities
video_predictions['raw_prediction'] = (raw_probabilities >= raw_selected_threshold).astype(int)
video_predictions['final_calibrated_prediction'] = (final_calibrated_probabilities >= calibrated_selected_threshold).astype(int)
video_predictions['final_temperature'] = final_temperature
video_predictions['calibrated_selected_threshold'] = calibrated_selected_threshold
video_predictions['uncertain'] = video_predictions['final_calibrated_probability'].between(0.4, 0.6, inclusive='both')
video_predictions.to_csv(CALIBRATED_PREDICTIONS_PATH, index=False)
calibrated_threshold_curve.to_csv(THRESHOLD_CURVE_PATH, index=False)

figure, axes = plt.subplots(1, 3, figsize=(15, 4))
for reliability, label, color in (
    (raw_reliability, 'raw', '#1f77b4'),
    (cross_fitted_reliability, 'cross-fitted calibrated', '#ff7f0e'),
    (final_fitted_reliability, 'final fitted calibrated', '#2ca02c'),
):
    nonempty = reliability.loc[reliability['count'].gt(0)]
    axes[0].plot(nonempty['mean_probability'], nonempty['empirical_frequency'], marker='o', label=label, color=color)
axes[0].plot([0, 1], [0, 1], '--', color='gray', label='perfect calibration')
axes[0].set(xlabel='Mean predicted probability', ylabel='Observed accident frequency', title='Reliability diagram', xlim=(0, 1), ylim=(0, 1))
axes[0].legend(fontsize=8)

axes[1].hist(raw_probabilities[labels == 0], bins=10, alpha=0.65, label='no accident', color='#1f77b4')
axes[1].hist(raw_probabilities[labels == 1], bins=10, alpha=0.65, label='accident', color='#d62728')
axes[1].axvline(raw_selected_threshold, color='black', linestyle='--', label=f'raw threshold {raw_selected_threshold:.2f}')
axes[1].set(xlabel='Raw video probability', ylabel='Videos', title='Raw probability distribution')
axes[1].legend(fontsize=8)

axes[2].plot(calibrated_threshold_curve['threshold'], calibrated_threshold_curve['f1'], label='F1')
axes[2].plot(calibrated_threshold_curve['threshold'], calibrated_threshold_curve['recall'], label='Recall')
axes[2].axvline(calibrated_selected_threshold, color='black', linestyle='--', label=f'deployment {calibrated_selected_threshold:.3f}')
axes[2].axhline(MINIMUM_ACCIDENT_RECALL, color='gray', linestyle=':', label=f'min recall {MINIMUM_ACCIDENT_RECALL:.2f}')
axes[2].set(xlabel='Calibrated threshold', ylabel='Metric', title='Threshold selection', xlim=(0.05, 0.95), ylim=(0, 1))
axes[2].legend(fontsize=8)
figure.tight_layout()
figure.savefig(RELIABILITY_PLOT_PATH, dpi=170)
plt.close(figure)

report = {
    'model': 'A2-MP ResNet18 frozen + mean-max pooling',
    'evaluation_scope': 'development calibration on the same fixed 120-video validation split; cross-fitted metrics are less optimistic than final-fitted metrics',
    'aggregation': selected_aggregation,
    'minimum_accident_recall_for_threshold_selection': MINIMUM_ACCIDENT_RECALL,
    'raw_selected_threshold_from_full_video_evaluation': raw_selected_threshold,
    'final_temperature_fitted_on_all_validation_videos': final_temperature,
    'calibrated_threshold_equivalent_to_raw_decision': calibrated_selected_threshold,
    'grid_selected_threshold_for_reference_only': grid_selected_threshold,
    'raw_metrics_at_raw_selected_threshold': raw_metrics_at_raw_threshold,
    'final_fitted_calibrated_metrics_at_selected_threshold': calibrated_metrics_at_selected_threshold,
    'brier_score_raw': float(brier_score_loss(labels, raw_probabilities)),
    'brier_score_cross_fitted_calibrated': float(brier_score_loss(labels, cross_fitted_probabilities)),
    'brier_score_final_fitted_calibrated': float(brier_score_loss(labels, final_calibrated_probabilities)),
    'ece_raw': raw_ece,
    'ece_cross_fitted_calibrated': cross_fitted_ece,
    'ece_final_fitted_calibrated': final_fitted_ece,
    'cross_fitted_temperatures': cross_fitted_temperatures,
    'uncertain_videos_at_0_4_to_0_6': int(video_predictions['uncertain'].sum()),
}
CALIBRATION_REPORT_PATH.write_text(json.dumps(report, indent=2), encoding='utf-8')

print('Raw full-MP4 metrics:')
print(raw_metrics_at_raw_threshold)
print('Final-fitted calibrated metrics at the decision-equivalent threshold:')
print(calibrated_metrics_at_selected_threshold)
display(pd.DataFrame([{
    'raw_brier': report['brier_score_raw'],
    'cross_fitted_calibrated_brier': report['brier_score_cross_fitted_calibrated'],
    'final_fitted_calibrated_brier': report['brier_score_final_fitted_calibrated'],
    'raw_ece': report['ece_raw'],
    'cross_fitted_calibrated_ece': report['ece_cross_fitted_calibrated'],
    'final_fitted_calibrated_ece': report['ece_final_fitted_calibrated'],
    'uncertain_videos': report['uncertain_videos_at_0_4_to_0_6'],
}]))
print(f'Calibrated predictions: {CALIBRATED_PREDICTIONS_PATH}')
print(f'Report: {CALIBRATION_REPORT_PATH}')
print(f'Plot: {RELIABILITY_PLOT_PATH}')

Raw full-MP4 metrics:
{'threshold': 0.4, 'accuracy': 0.6916666666666667, 'precision': 0.6419753086419753, 'recall': 0.8666666666666667, 'f1': 0.7375886524822695, 'roc_auc': 0.7402777777777777, 'pr_auc': 0.7226701049305261, 'brier_score': 0.20956536408696297, 'confusion_matrix': [[31, 29], [8, 52]]}
Final-fitted calibrated metrics at the decision-equivalent threshold:
{'threshold': 0.4285218762661993, 'accuracy': 0.6916666666666667, 'precision': 0.6419753086419753, 'recall': 0.8666666666666667, 'f1': 0.7375886524822695, 'roc_auc': 0.7402777777777777, 'pr_auc': 0.7226701049305261, 'brier_score': 0.20770098642065835, 'confusion_matrix': [[31, 29], [8, 52]]}


,raw_brier,cross_fitted_calibrated_brier,final_fitted_calibrated_brier,raw_ece,cross_fitted_calibrated_ece,final_fitted_calibrated_ece,uncertain_videos
0,0.209565,0.210787,0.207701,0.107837,0.053961,0.050828,40


Calibrated predictions: P:\NexarCollisionData\inference_v2\a2_multipos_calibrated_validation_predictions.csv
Report: P:\NexarCollisionData\inference_v2\a2_multipos_calibration_report.json
Plot: P:\NexarCollisionData\inference_v2\a2_multipos_reliability_diagram.png
